# Second-Order Methods and Curvature Information

- Module: 01 Optimization
- Topic: Newton-style updates and Hessian geometry
- Estimated runtime: 10 minutes
- Prerequisites: Hessians, matrix solves, positive definiteness, gradient descent
- Expected outputs: trajectory comparisons on a curved valley, convergence summaries, and a saddle-point Newton failure experiment

## Learning goals

- See how Newton's method rescales directions using the local Hessian.
- Compare first-order and second-order behavior on a curved, anisotropic surface.
- Understand why an indefinite Hessian can make an undamped Newton step move toward a saddle instead of away from it.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().parents[2]
src_dir = repo_root / "modules" / "01-optimization" / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from optim_utils import (
    adagrad,
    adam,
    gradient_descent,
    make_quadratic,
    make_shifted_least_squares,
    momentum_descent,
    newton_method,
    plot_convergence,
    plot_trajectories,
    rmsprop,
    rosenbrock_objective,
    saddle_objective,
    stochastic_gradient_descent,
    summarize_history,
)

np.set_printoptions(precision=4, suppress=True)


## A curved valley: the Rosenbrock objective

The Rosenbrock function is not difficult because of many local minima. It is difficult because the minimizer sits inside a narrow curved valley, which makes first-order methods struggle to choose a direction that is both stable and fast.

In [ ]:
objective = rosenbrock_objective(a=1.0, b=18.0)
start = np.array([-1.4, 1.6], dtype=float)
gd_history = gradient_descent(objective, start=start, learning_rate=0.02, steps=60)
gd_history["name"] = "GD"
newton_history = newton_method(objective, start=start, steps=10, damping=1.0, regularization=1e-6)
newton_history["name"] = "Newton"
[summarize_history(history, objective) | {"name": history["name"]} for history in [gd_history, newton_history]]

## Trajectories in the valley

Gradient descent follows the local slope and tends to bounce across the valley walls. Newton's method uses the Hessian to rescale coordinates, so it can take much straighter steps toward the minimizer when the Hessian is well behaved.

In [ ]:
fig, _ = plot_trajectories(
    objective,
    [gd_history, newton_history],
    x_range=(-1.8, 1.8),
    y_range=(-0.5, 2.4),
    title="Gradient descent versus Newton on Rosenbrock",
    levels=24,
)
plt.show()
plt.close(fig)

## Convergence comparison

Near a well-behaved minimizer, Newton's method often shows much faster local convergence than a first-order method. The plots make that curvature advantage visible even with a small number of iterations.

In [ ]:
fig, _ = plot_convergence([gd_history, newton_history], title="First-order versus second-order")
plt.show()
plt.close(fig)

## Failure mode: undamped Newton at a saddle

Newton solves $H(x_k) p_k = 
abla f(x_k)$ and then moves to $x_{k+1} = x_k - p_k$. When the Hessian is indefinite, that linear solve can point toward a saddle rather than toward a minimizer, because the method is not enforcing descent by itself.

In [ ]:
saddle = saddle_objective()
saddle_start = np.array([0.2, 0.8], dtype=float)
newton_saddle = newton_method(saddle, start=saddle_start, steps=6)
newton_saddle["name"] = "Newton on saddle"
gd_saddle = gradient_descent(saddle, start=saddle_start, learning_rate=0.08, steps=6)
gd_saddle["name"] = "GD on saddle"

fig, _ = plot_trajectories(
    saddle,
    [gd_saddle, newton_saddle],
    x_range=(-1.0, 1.0),
    y_range=(-1.2, 1.2),
    title="Newton can be attracted to a saddle",
    levels=18,
)
plt.show()
plt.close(fig)

[summarize_history(history, saddle) | {"name": history["name"]} for history in [gd_saddle, newton_saddle]]

## Interpretation

On the saddle surface, Newton jumps to the stationary point at the origin because solving with the indefinite Hessian treats that stationary point as acceptable. Gradient descent does not settle there because the negative-curvature direction keeps pushing it away. This is why practical second-order methods usually add damping, trust regions, or line searches.

## Summary

Second-order information can dramatically improve optimization in curved valleys, but only when the Hessian is numerically and geometrically trustworthy. Indefinite curvature is the central caution sign.

## Next steps

- Add damping to Newton's method and compare the path near the saddle.
- Replace Rosenbrock with an ill-conditioned quadratic and explain why Newton solves it in one step.
- Investigate quasi-Newton updates that approximate Hessians without forming them exactly.